# Adaptive MobileViT U‑Net (Hybrid CNN–Transformer)

This notebook implements an adaptive hybrid U‑Net that injects a small MobileViT‑style
transformer block into decoder stages. The block uses patch embedding + a lightweight
transformer encoder (Keras MultiHeadAttention) and an adaptive gating mechanism to
control how much global attention contributes vs local convolutional features.

Replace the placeholder data loading with ISIC 2020 or Cityscapes loaders to train on
real data. The notebook includes model build / demo cells for a quick sanity check.

In [ ]:
# Imports and setup
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
import numpy as np
tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
# Basic conv / encoder / decoder helpers
def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    return x

def down_block(x, filters):
    c = conv_block(x, filters)
    p = layers.MaxPooling2D()(c)
    return c, p

def up_block(x, skip, filters):
    x = layers.Conv2DTranspose(filters, 2, strides=2, padding='same')(x)
    x = layers.Concatenate()([x, skip])
    x = conv_block(x, filters)
    return x

In [ ]:
# MobileViT-style patch embedding + lightweight transformer encoder (implemented simply)
def patch_embedding(x, patch_size=2, embed_dim=64):
    # x: (B,H,W,C) -> unfold into non-overlapping patches of size patch_size
    H = tf.shape(x)[1]
    W = tf.shape(x)[2]
    # Use a conv with stride=patch_size to create patch vectors
    x = layers.Conv2D(embed_dim, kernel_size=patch_size, strides=patch_size, padding='valid')(x)
    # now (B, H/ps, W/ps, embed_dim) -> flatten to sequence later in transformer block
    return x

def lightweight_transformer_on_patches(x, num_heads=4, mlp_dim=128, dropout=0.0):
    # x: (B, H_p, W_p, embed_dim) -> treat as sequence of length H_p*W_p
    shape = tf.shape(x)
    B, Hp, Wp, C = shape[0], shape[1], shape[2], shape[3]
    seq = layers.Reshape((Hp * Wp, C))(x)
    # MHSA (queries/keys/values all from seq)
    attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=max(1, C // num_heads), dropout=dropout)(seq, seq)
    seq = layers.Add()([seq, attn])
    seq = layers.LayerNormalization(epsilon=1e-6)(seq)
    # MLP
    ff = layers.Dense(mlp_dim, activation='gelu')(seq)
    ff = layers.Dense(C)(ff)
    seq = layers.Add()([seq, ff])
    seq = layers.LayerNormalization(epsilon=1e-6)(seq)
    # reshape back to (B, Hp, Wp, C)
    out = layers.Reshape((Hp, Wp, C))(seq)
    return out

def mobilevit_block(x, patch_size=2, embed_dim=64, num_heads=4, mlp_dim=128):
    # Local representation (conv)
    local_feat = layers.Conv2D(embed_dim, 3, padding='same', activation='relu')(x)
    # Patch embedding over local features
    patches = patch_embedding(local_feat, patch_size=patch_size, embed_dim=embed_dim)
    # Transformer on patches
    transformed = lightweight_transformer_on_patches(patches, num_heads=num_heads, mlp_dim=mlp_dim)
    # Upsample transformed patches to original resolution by pixel shuffle-like expansion
    # naive resize using nearest/neighbour for simplicity
    Hp = tf.shape(transformed)[1]
    Wp = tf.shape(transformed)[2]
    # resize to match local_feat spatial dims
    transformed_up = tf.image.resize(transformed, (tf.shape(local_feat)[1], tf.shape(local_feat)[2]), method='bilinear')
    # fuse local and global features, then return
    fused = layers.Concatenate()([local_feat, transformed_up])
    fused = layers.Conv2D(tf.shape(x)[-1], 1, padding='same', activation='relu')(fused)
    # adaptive gate (learn how much to use fused/global info)
    gate = layers.Conv2D(1, 1, activation='sigmoid')(fused)
    out = layers.Multiply()([fused, gate])
    out = layers.Add()([x, out])
    return out

In [ ]:
# Build a U-Net with MobileViT blocks in the decoder
def build_mobilevit_unet(input_shape=(128,128,1), base_filters=32, patch_size=2, embed_dim=64):
    inputs = Input(input_shape)
    c1, p1 = down_block(inputs, base_filters)
    c2, p2 = down_block(p1, base_filters*2)
    c3, p3 = down_block(p2, base_filters*4)
    c4, p4 = down_block(p3, base_filters*8)
    b = conv_block(p4, base_filters*16)
    # decoder with MobileViT blocks injected
    u6 = layers.Conv2DTranspose(base_filters*8, 2, strides=2, padding='same')(b)
    u6 = layers.Concatenate()([u6, c4])
    u6 = mobilevit_block(u6, patch_size=patch_size, embed_dim=embed_dim)
    u6 = conv_block(u6, base_filters*8)

    u7 = layers.Conv2DTranspose(base_filters*4, 2, strides=2, padding='same')(u6)
    u7 = layers.Concatenate()([u7, c3])
    u7 = mobilevit_block(u7, patch_size=patch_size, embed_dim=embed_dim)
    u7 = conv_block(u7, base_filters*4)

    u8 = layers.Conv2DTranspose(base_filters*2, 2, strides=2, padding='same')(u7)
    u8 = layers.Concatenate()([u8, c2])
    u8 = mobilevit_block(u8, patch_size=patch_size, embed_dim=embed_dim)
    u8 = conv_block(u8, base_filters*2)

    u9 = layers.Conv2DTranspose(base_filters, 2, strides=2, padding='same')(u8)
    u9 = layers.Concatenate()([u9, c1])
    u9 = mobilevit_block(u9, patch_size=patch_size, embed_dim=embed_dim)
    u9 = conv_block(u9, base_filters)

    outputs = layers.Conv2D(1, 1, activation='sigmoid')(u9)
    model = Model(inputs, outputs, name='MobileViT_UNet')
    return model

In [ ]:
# Losses and metrics
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2.*intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

def iou(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

In [ ]:
# Quick sanity build + forward pass on dummy data
model = build_mobilevit_unet(input_shape=(128,128,1), base_filters=16, patch_size=2, embed_dim=64)
model.summary()
x = np.random.rand(1,128,128,1).astype('float32')
y = model.predict(x)
print('Output shape:', y.shape)